# 研究方向趋势扫描

> 查看某方向近 5 年热度、头部期刊、高被引论文和关键词变化

**场景**: 研究者想了解某个方向近 5 年的发展趋势：发文量变化、头部期刊分布、高被引论文、关键词演变。可用 meta-search 分年统计支撑。

**使用接口**: meta-search

**预估调用量**: ~10–25 次 API 调用 / 一次扫描

---


## Step 1: 环境准备

安装依赖并配置 API Token


In [ ]:
!pip install httpx pandas
import os
os.environ["SCIVERSE_API_TOKEN"] = "sv-your-token-here"  # 替换为你的真实值


## Step 2: 按年统计发文量

用 meta-search 分年查询，统计每年的发文数量


In [ ]:
import os
import asyncio
import httpx
import pandas as pd

BASE = "https://api.sciverse.space"
TOKEN = os.environ["SCIVERSE_API_TOKEN"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def count_by_year(query: str, year: int) -> int:
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(
            f"{BASE}/meta-search", headers=HEADERS,
            json={
                "query": query,
                "filters": [
                    {"field": "publication_published_year", "operator": "FILTER_OP_EQ", "value": year}
                ],
                "page": 1, "page_size": 1
            }
        )
        resp.raise_for_status()
        return resp.json()["total_count"]

async def trend_scan(query: str, start_year: int = 2020, end_year: int = 2024):
    tasks = [count_by_year(query, y) for y in range(start_year, end_year + 1)]
    counts = await asyncio.gather(*tasks)
    return list(zip(range(start_year, end_year + 1), counts))

trend = asyncio.run(trend_scan("large language model"))
df = pd.DataFrame(trend, columns=["year", "count"])
print(df.to_string(index=False))


## Step 3: 查找高被引论文和头部期刊

按引用数排序，找出各年最具影响力的论文


In [ ]:
async def top_cited_papers(year: int, top_n: int = 5):
    """查找某年度高被引论文（按引用数排序时不传 query）"""
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(
            f"{BASE}/meta-search", headers=HEADERS,
            json={
                "filters": [
                    {"field": "publication_published_year", "operator": "FILTER_OP_EQ", "value": year}
                ],
                "sort": [{"field": "citation_count", "order": "SORT_ORDER_DESC"}],
                "page": 1, "page_size": top_n
            }
        )
        resp.raise_for_status()
        return resp.json()["results"]

async def main():
    for year in [2022, 2023, 2024]:
        papers = await top_cited_papers(year)
        print(f"\
=== {year} Top Cited ===")
        for p in papers:
            venue = p.get("publication_venue_name", "N/A")
            cites = p.get("citation_count", 0)
            print(f"  [{cites} cites] {p['title'][:60]} ({venue})")

asyncio.run(main())


## 注意事项

- meta-search 的 sort 和 query 不能同时传；按引用数排序时只传 filters + sort
- 分年查询可并发执行（asyncio.gather）提高效率
- total_count 可直接作为当年发文量，无需拉取全部结果
- 可进一步统计 venue 分布、作者网络等


---

## 下一步

- [申请 API Token](https://sciverse.opendatalab.com/docs#auth)
- [查看完整 API 文档](https://sciverse.opendatalab.com/docs#sciverse/api)
- [更多 Cookbook 案例](https://sciverse.opendatalab.com/docs#cookbook)
